In [ ]:
# ---------------------------------------------------------
# 1. Use the recurrent model selected on validation data
# ---------------------------------------------------------

# The official test set is not used to choose the model for the agent.
if selected_recurrent_model_name == best_gru_config["name"]:
    selected_prediction = gru_test_pred
elif selected_recurrent_model_name == best_lstm_config["name"]:
    selected_prediction = lstm_test_pred
else:
    raise ValueError(
        "Validation-selected model does not match the finalized LSTM/GRU models."
    )

print("Validation-selected recurrent model:", selected_recurrent_model_name)


# ---------------------------------------------------------
# 2. Create one decision record for each test engine
# ---------------------------------------------------------

agent_input = pd.DataFrame({
    "unit_id": test_unit_ids,
    "predicted_rul": selected_prediction,

    # This is only a practical warning signal, not a calibrated confidence interval.
    "model_disagreement": np.abs(
        lstm_test_pred - gru_test_pred
    ),
})


# ---------------------------------------------------------
# 3. Convert predicted RUL into illustrative risk groups
# ---------------------------------------------------------

def classify_risk(predicted_rul: float) -> str:
    if predicted_rul <= 15:
        return "Critical"
    if predicted_rul <= 30:
        return "High"
    if predicted_rul <= 60:
        return "Medium"
    return "Low"


agent_input["risk_tier"] = (
    agent_input["predicted_rul"]
    .map(classify_risk)
)


# ---------------------------------------------------------
# 4. Flag predictions that deserve human review
# ---------------------------------------------------------

DISAGREEMENT_THRESHOLD = 10.0

agent_input["review_required"] = (
    agent_input["model_disagreement"] >= DISAGREEMENT_THRESHOLD
)


# ---------------------------------------------------------
# 5. Calculate maintenance urgency
# ---------------------------------------------------------

# Risk tier controls the broad ordering. Within a tier, lower RUL ranks first.
RISK_WEIGHTS = {
    "Critical": 300,
    "High": 200,
    "Medium": 100,
    "Low": 0,
}

agent_input["urgency_score"] = (
    agent_input["risk_tier"].map(RISK_WEIGHTS)
    + 100.0 / (agent_input["predicted_rul"] + 1.0)
)


# ---------------------------------------------------------
# 6. Rank the fleet from most urgent to least urgent
# ---------------------------------------------------------

maintenance_queue = (
    agent_input
    .sort_values(
        by=["urgency_score", "predicted_rul"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

maintenance_queue["queue_position"] = (
    np.arange(1, len(maintenance_queue) + 1)
)


# ---------------------------------------------------------
# 7. Recommend a maintenance action
# ---------------------------------------------------------

# Maintenance action and model-review status are kept separate.
# A Critical engine can therefore be scheduled immediately AND flagged for review.
def recommend_action(risk_tier: str) -> str:
    if risk_tier == "Critical":
        return "Schedule immediately"
    if risk_tier == "High":
        return "Schedule soon"
    if risk_tier == "Medium":
        return "Monitor closely"
    return "Continue normal monitoring"


maintenance_queue["recommended_action"] = (
    maintenance_queue["risk_tier"]
    .map(recommend_action)
)

maintenance_queue["review_note"] = np.where(
    maintenance_queue["review_required"],
    "Review model disagreement",
    "",
)

display(maintenance_queue.head(20))

maintenance_queue.to_csv(
    REPORTS_DIR / "maintenance_queue.csv",
    index=False,
)


# ---------------------------------------------------------
# 8. Capacity analysis
# ---------------------------------------------------------

# Only High and Critical engines are eligible for maintenance.
# Extra capacity is left unused rather than maintaining lower-risk engines early.
eligible_queue = (
    maintenance_queue[
        maintenance_queue["risk_tier"]
        .isin(["Critical", "High"])
    ]
    .copy()
)

risk_counts = maintenance_queue["risk_tier"].value_counts()

total_critical = int(risk_counts.get("Critical", 0))
total_high = int(risk_counts.get("High", 0))
total_medium = int(risk_counts.get("Medium", 0))
total_low = int(risk_counts.get("Low", 0))
total_high_or_critical = total_critical + total_high

capacity_summary = []

for capacity in [5, 10, 15, 20, 30]:
    scheduled = eligible_queue.head(capacity)

    capacity_summary.append({
        "capacity": int(capacity),
        "total_critical": total_critical,
        "total_high": total_high,
        "total_medium": total_medium,
        "total_low": total_low,
        "total_high_or_critical": total_high_or_critical,
        "scheduled_engines": int(len(scheduled)),
        "unused_capacity": int(capacity - len(scheduled)),
        "critical_scheduled": int(
            (scheduled["risk_tier"] == "Critical").sum()
        ),
        "high_scheduled": int(
            (scheduled["risk_tier"] == "High").sum()
        ),
        "high_or_critical_scheduled": int(
            scheduled["risk_tier"]
            .isin(["Critical", "High"])
            .sum()
        ),
        "manual_reviews": int(
            scheduled["review_required"].sum()
        ),
        "maximum_scheduled_predicted_rul": (
            float(scheduled["predicted_rul"].max())
            if not scheduled.empty
            else np.nan
        ),
    })

capacity_summary = pd.DataFrame(capacity_summary)
display(capacity_summary)

capacity_summary.to_csv(
    REPORTS_DIR / "maintenance_capacity_analysis.csv",
    index=False,
)
